# TEM-Style 4x4 Training on Colab

This notebook trains the 4x4 next-state task using the original TEM-style sequential loop: observation remapping, path integration, Hebbian memory, and a supervised next-state head.

In [ ]:
import importlib
import subprocess
import sys

def ensure_package(package_name):
    try:
        importlib.import_module(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

for package in ["scipy", "tensorboard"]:
    ensure_package(package)

print("Dependencies ready.")

In [ ]:
import os

repo_dir = "/content/torch_tem"
if not os.path.exists(repo_dir):
    !git clone https://github.com/YifeiCAO/torch_tem.git {repo_dir}
else:
    print(f"Repository already exists at {repo_dir}")
    !git -C {repo_dir} pull

%cd /content/torch_tem

In [ ]:
import importlib
import torch

import train_2d_tem_style
importlib.reload(train_2d_tem_style)

train = train_2d_tem_style.train
load_model = train_2d_tem_style.load_model

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
TRAIN_STEPS = 2000
BATCH_SIZE = 16
ROLLOUT_LENGTH = 20
SUPERVISED_WEIGHT = 1.0
TEM_WEIGHT = 1.0
SEED = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHECKPOINT_PATH = "/content/torch_tem/tem_2d_style.pt"

In [ ]:
predictor = train(
    train_steps=TRAIN_STEPS,
    batch_size=BATCH_SIZE,
    rollout_length=ROLLOUT_LENGTH,
    supervised_weight=SUPERVISED_WEIGHT,
    tem_weight=TEM_WEIGHT,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every=250,
    seed=SEED,
    device=DEVICE,
)

print(f"Saved checkpoint to {CHECKPOINT_PATH}")

In [ ]:
loaded_predictor, train_config, _ = load_model(
    checkpoint_path=CHECKPOINT_PATH,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

print("Loaded checkpoint.")
print(train_config)